In [ ]:
from pyspark.sql.functions import (
    col, trim, when, max as spark_max, explode, lit, from_json
)

from delta.tables import DeltaTable

tabela_nome = "silver_contas"

if spark.catalog.tableExists("controle_pipeline"):
        df_controle = spark.table("controle_pipeline").filter(col("tabela") == tabela_nome)

        if df_controle.count() > 0: 
            ultimo_processamento = df_controle.select("ultimo_ts").collect()[0][0]
        else:
            ultimo_processamento = None
else: 
    ultimo_processamento = None

In [ ]:
nome_tabela = "contas"
caminho_bronze = f"Files/bronze/{nome_tabela}"
df_bronze = spark.read.format("delta").load(caminho_bronze)

if ultimo_processamento:
    df_bronze = df_bronze.filter(
        col("ingestion_ts") > ultimo_processamento
    )

if df_bronze.rdd.isEmpty():
    print("Sem dados novos para processar.")
    dbutils.notebook.exit("No data")

In [ ]:
from pyspark.sql.types import *

schema = StructType([
    StructField("bairro", StringType()),
    StructField("bloqueado", StringType()),
    StructField("bol_instr1", StringType()),
    StructField("bol_instr2", StringType()),
    StructField("bol_instr3", StringType()),
    StructField("bol_instr4", StringType()),
    StructField("bol_sn", StringType()),
    StructField("cCnpjInstFinanc", StringType()),
    StructField("cCodCCInt", StringType()),
    StructField("cEstabelecimento", StringType()),
    StructField("cTipoCartao", StringType()),
    StructField("cancinstr", StringType()),
    StructField("cep", LongType()),
    StructField("cidade", StringType()),
    StructField("cnab_esp", StringType()),
    StructField("cobr_esp", StringType()),
    StructField("cobr_sn", StringType()),
    StructField("codigo_agencia", StringType()),
    StructField("codigo_banco", StringType()),
    StructField("codigo_pais", StringType()),
    StructField("complemento", StringType()),
    StructField("data_alt", StringType()),
    StructField("data_inc", StringType()),
    StructField("ddd", StringType()),
    StructField("descricao", StringType()),
    StructField("dias_rcomp", LongType()),
    StructField("email", StringType()),
    StructField("endereco", StringType()),
    StructField("estado", StringType()),
    StructField("hora_alt", StringType()),
    StructField("hora_inc", StringType()),
    StructField("importado_api", StringType()),
    StructField("inativo", StringType()),
    StructField("modalidade", StringType()),
    StructField("nCodCC", LongType()),
    StructField("nao_fluxo", StringType()),
    StructField("nao_resumo", StringType()),
    StructField("nome_gerente", StringType()),
    StructField("numero", StringType()),
    StructField("numero_conta_corrente", StringType()),
    StructField("observacao", StringType()),
    StructField("pdv_bandeira", StringType()),
    StructField("pdv_categoria", StringType()),
    StructField("pdv_cod_adm", LongType()),
    StructField("pdv_dias_venc", LongType()),
    StructField("pdv_enviar", StringType()),
    StructField("pdv_limite_pacelas", LongType()),
    StructField("pdv_num_parcelas", LongType()),
    StructField("pdv_sincr_analitica", StringType()),
    StructField("pdv_taxa_adm", LongType()),
    StructField("pdv_taxa_loja", LongType()),
    StructField("pdv_tipo_tef", LongType()),
    StructField("per_juros", LongType()),
    StructField("per_multa", LongType()),
    StructField("pix_sn", StringType()),
    StructField("saldo_data", StringType()),
    StructField("saldo_inicial", LongType()),
    StructField("telefone", StringType()),
    StructField("tipo", StringType()),
    StructField("tipo_conta_corrente", StringType()),
    StructField("user_alt", StringType()),
    StructField("user_inc", StringType()),
    StructField("valor_limite", LongType()),
    StructField("empresa", StringType()) 

])

In [ ]:
df_bronze = df_bronze.withColumn("json", from_json(col("raw_json"), schema))
df_silver = df_bronze.select("json.*", "ingestion_ts")

In [ ]:
for c, t in df_silver.dtypes:
    if t == "string":
        df_silver = df_silver.withColumn(
            c,
            when(trim(col(c)) == "", None)
            .otherwise(trim(col(c)))
        )

In [ ]:
df_silver = df_silver.dropDuplicates(["nCodCC", "empresa"])

In [ ]:
if spark.catalog.tableExists("silver_contas"):

    delta_table = DeltaTable.forName(spark, "silver_contas")

    (delta_table.alias("t")
            .merge(
                df_silver.alias("s"),
                """
                t.nCodCC = s.nCodCC
                AND t.empresa = s.empresa
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()  
    )

else: 
    df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_contas")

In [ ]:
novo_max_ts = df_bronze.agg(
    spark_max("ingestion_ts").alias("max_ts")).collect()[0]["max_ts"]

df_update = spark.createDataFrame(
    [(tabela_nome, novo_max_ts)],
    ["tabela", "ultimo_ts"]
)

if spark.catalog.tableExists("controle_pipeline"):

    delta_ctrl = DeltaTable.forName(spark, "controle_pipeline")

    (delta_ctrl.alias("t")
        .merge(
            df_update.alias("s"),
            "t.tabela = s.tabela"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else: 
    df_update.write.format("delta").mode("overwrite").saveAsTable("controle_pipeline")